In [ ]:
# Извлечение GPS координат
print("Извлечение GPS координат...")
start_coords = gps_extractor.extract_from_video_metadata(video_path)

if start_coords is None:
    print("GPS координаты не найдены в метаданных")
    print("Используйте координаты по умолчанию или укажите свои:")
    start_coords = (55.7558, 37.6173)  # Москва (пример)
    print(f"Используются координаты: {start_coords}")
else:
    print(f"✓ Начальные координаты: {start_coords}")

end_coords = None  # Можно указать конечные координаты
speed_kmh = 50.0  # Скорость для интерполяции координат


In [ ]:
# Обработка видео - распознавание всех объектов
print("=" * 70)
print("ОБРАБОТКА ВИДЕО")
print("=" * 70)

cap = cv2.VideoCapture(video_path)

# Параметры видео
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"Параметры видео:")
print(f"  Разрешение: {width}x{height}")
print(f"  FPS: {fps:.2f}")
print(f"  Кадров: {total_frames}")
print(f"  Длительность: {total_frames/fps:.2f} сек")

# Обрабатываем каждый N-й кадр для ускорения
frame_skip = int(fps)  # 1 кадр в секунду
frame_count = 0
processed_count = 0

print("\nНачало обработки...")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # Обрабатываем каждый N-й кадр
    if frame_count % frame_skip == 0:
        # Координаты для текущего кадра
        if end_coords:
            frame_coords = gps_extractor.calculate_position_by_time(
                start_coords, end_coords, frame_count, total_frames
            )
        else:
            frame_coords = gps_extractor.interpolate_coordinates(
                frame_count, fps, start_coords, speed_kmh=speed_kmh
            )
        
        # Распознавание дорожных знаков (155 классов RTSD!)
        signs = sign_detector.detect(frame)
        for sign in signs:
            sign.coordinates = gps_extractor.bbox_to_gps(
                sign.bbox, frame_coords, width, height
            )
            sign.timestamp = datetime.now()
            passport.add_object(sign, frame_coords, frame_count)
        
        # Распознавание разметки и полос
        lane_info = marking_detector.detect_lanes(frame)
        if lane_info.lane_count > 0:
            passport.add_lane_info(lane_info)
        
        for marking in lane_info.lane_markings:
            marking.timestamp = datetime.now()
            passport.add_marking(marking, frame_coords)
        
        # Пешеходные переходы
        crosswalks = marking_detector.detect_crosswalk(frame)
        for crosswalk in crosswalks:
            crosswalk.timestamp = datetime.now()
            passport.add_marking(crosswalk, frame_coords)
        
        # Светофоры
        traffic_lights = traffic_light_detector.detect(frame)
        for light in traffic_lights:
            light.coordinates = gps_extractor.bbox_to_gps(
                light.bbox, frame_coords, width, height
            )
            light.timestamp = datetime.now()
            passport.add_object(light, frame_coords, frame_count)
        
        # Уличные фонари
        street_lights = street_light_detector.detect(frame)
        for light in street_lights:
            light.coordinates = gps_extractor.bbox_to_gps(
                light.bbox, frame_coords, width, height
            )
            light.timestamp = datetime.now()
            passport.add_object(light, frame_coords, frame_count)
        
        # Ограждения
        fences = fence_detector.detect(frame)
        for fence in fences:
            fence.coordinates = gps_extractor.bbox_to_gps(
                fence.bbox, frame_coords, width, height
            )
            fence.timestamp = datetime.now()
            passport.add_object(fence, frame_coords, frame_count)
        
        processed_count += 1
        if processed_count % 10 == 0:
            print(f"Обработано: {processed_count}, объектов: {len(passport.objects)}, разметки: {len(passport.markings)}")
    
    frame_count += 1

cap.release()

print(f"\n{'='*70}")
print(f"ОБРАБОТКА ЗАВЕРШЕНА")
print(f"{'='*70}")
print(f"Обработано кадров: {processed_count}")
print(f"Обнаружено объектов: {len(passport.objects)}")
print(f"Обнаружено разметки: {len(passport.markings)}")
print(f"Информация о полосах: {len(passport.lane_infos)}")


In [ ]:
# СОЗДАНИЕ ВИДЕО С ВИЗУАЛИЗАЦИЕЙ ВСЕХ ОБЪЕКТОВ
# На видео будут показаны: дорожные знаки, разметка, светофоры, фонари и т.д.

print("=" * 70)
print("СОЗДАНИЕ ВИДЕО С ВИЗУАЛИЗАЦИЕЙ")
print("=" * 70)
print("На видео будут отмечены:")
print("  ✓ Дорожные знаки (155 классов RTSD)")
print("  ✓ Разметка (линии, пешеходные переходы)")
print("  ✓ Светофоры")
print("  ✓ Уличные фонари")
print("  ✓ Ограждения")
print()

cap = cv2.VideoCapture(video_path)
output_path = "road_analysis_output.mp4"

# Создаем видео с тем же FPS и разрешением
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

# Храним объекты с номерами кадров
objects_by_frame = {}
for i, obj in enumerate(passport.objects):
    # Распределяем объекты по кадрам (упрощенно)
    # В реальности нужно хранить номер кадра вместе с объектом
    frame_num = (i * frame_skip) % total_frames
    if frame_num not in objects_by_frame:
        objects_by_frame[frame_num] = []
    objects_by_frame[frame_num].append(obj)

markings_by_frame = {}
for i, marking in enumerate(passport.markings):
    frame_num = (i * frame_skip) % total_frames
    if frame_num not in markings_by_frame:
        markings_by_frame[frame_num] = []
    markings_by_frame[frame_num].append(marking)

frame_count = 0
print("Обработка видео с визуализацией...")

cap.set(cv2.CAP_PROP_POS_FRAMES, 0)  # Возврат к началу

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # Получаем объекты для текущего кадра
    current_objects = objects_by_frame.get(frame_count, [])
    current_markings = markings_by_frame.get(frame_count, [])
    
    # Рисуем объекты на кадре
    for obj in current_objects:
        x1, y1, x2, y2 = obj.bbox
        
        # Цвета по типам объектов
        color_map = {
            'дорожный_знак': (255, 0, 0),      # Синий
            'светофор': (0, 0, 255),           # Красный
            'фонарь': (0, 255, 255),           # Желтый
            'ограждение': (128, 128, 128),     # Серый
        }
        color = color_map.get(obj.object_type, (0, 255, 0))  # Зеленый по умолчанию
        
        # Рисуем прямоугольник вокруг объекта
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        
        # Добавляем подпись
        label = f"{obj.class_name} ({obj.confidence:.2f})"
        cv2.putText(frame, label, (x1, y1 - 10), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    
    # Рисуем разметку
    for marking in current_markings:
        if marking.marking_type == 'пешеходный_переход':
            # Пешеходный переход - прямоугольник
            cv2.rectangle(frame, marking.start_point, marking.end_point, 
                         (0, 255, 0), 2)
            cv2.putText(frame, "Пешеходный переход", 
                       (marking.start_point[0], marking.start_point[1] - 5),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)
        else:
            # Линии разметки
            cv2.line(frame, marking.start_point, marking.end_point, 
                    (255, 165, 0), 2)  # Оранжевый
    
    out.write(frame)
    frame_count += 1
    
    if frame_count % 100 == 0:
        print(f"Обработано кадров для видео: {frame_count}/{total_frames}")

cap.release()
out.release()

print(f"\n✓ Видео с визуализацией сохранено: {output_path}")
print(f"  Все объекты отмечены на видео:")
print(f"    - Дорожные знаки (синие рамки)")
print(f"    - Светофоры (красные рамки)")
print(f"    - Фонари (желтые рамки)")
print(f"    - Разметка (оранжевые линии)")
print(f"    - Пешеходные переходы (зеленые прямоугольники)")


In [ ]:
# Создание паспорта дороги
print("=" * 70)
print("ПАСПОРТ ДОРОГИ")
print("=" * 70)

df = passport.to_dataframe()

print("\nПервые 20 записей:")
print(df.head(20).to_string(index=False))

# Сохранение паспорта
passport.to_csv("road_passport.csv")
passport.to_json("road_passport.json")

print(f"\n✓ Паспорт сохранен:")
print(f"  - road_passport.csv")
print(f"  - road_passport.json")
